In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Policy gradient I: improving a policy directly

## Where this session fits

> **Part A learned values and read the policy off them with an argmax. Part B learns the policy itself, as a network trained by gradient ascent.**

- **Where Part A left us**
    - the argmax needs a short list of actions and gives a greedy (deterministic) policy
    - Part A's last slide proposed a network that outputs the policy; it never built one
- **This session** (Part B, session 1 of 4)
    - the network: what a policy network outputs
    - the training signal: the gradient of the expected return with respect to the network's parameters $\theta$
    - the **policy gradient theorem**, which gives that gradient from sampled episodes, without the environment's probabilities
- **Series goal:** choose good actions by trial and error, from small tables to robots and language models


**By the end you can**

1. say what a policy network outputs, for discrete and for continuous actions
2. say what the **score** of an action is, and where it comes from
3. write the objective $J(\theta)$ and say why its gradient is not obvious
4. state the policy gradient theorem, and say why the transition probabilities $\transp$ do not appear
5. say what one update does to the action probabilities
6. state the zero-mean score lemma
7. say what a single sampled estimate looks like on FrozenLake, and why the estimator is on-policy


# A policy as a network

**Why this section?** Part A ended at the limits of the argmax, so Part B learns the policy directly. First: what does a policy network output?

> **The policy network outputs a probability distribution over actions, and the agent samples its action from it.**

- the policy $\pi_\theta(\act \mid \state) = \pr{\actseq_\tt = \act \mid \stateseq_\tt = \state}$ is a *probability distribution* over actions, given the state
    - a *density* when the actions are continuous
    - $\theta$ are the network's parameters
- the network outputs the numbers that define the distribution, and the agent **samples** its action from it
    - **discrete actions:** one probability per action (the last layer is a softmax)
    - **continuous actions:** the mean and spread of a Gaussian (a bell-shaped distribution)
    - Microduck's policy is of the second kind: continuous motor targets
- **randomness is built in:** the policy samples its action, so exploration needs no $\epsilon$-greedy rule


# The running example: slippery FrozenLake

**Why this section?** We use one example throughout Part B, so that each session builds on the last. It is the grid of Part A, with a small step cost.

> **Reach the goal G without falling into a hole H, on a grid where the move you ask for happens only a third of the time.**

<img src="images/fl_map.png" width=68%>

- 16 states and 4 moves (left, down, right, up)
- **every move costs $0.01$** (a reward of $-0.01$), and entering G pays $+1$
- an episode ends at G or at a hole
- its **return** is the sum of its rewards
- Part A had no step cost: the small cost only makes shorter routes better


**A policy on FrozenLake: the uniform policy**

> **The uniform policy picks each of the 4 moves with probability $\frac14$ in every state. It is the starting point for training.**

<img src="images/fl_episodes.png" width=84%>

- three episodes sampled from the uniform policy
    - left: two moves, then a hole: return $2 \times (-0.01) = -0.02$
    - middle: nine moves that wander, then a hole: return $9 \times (-0.01) = -0.09$
    - right: nine moves and the last one enters G: return $8 \times (-0.01) + 0.99 = +0.91$
    - a circle marks a move that left the agent where it was (it walked into a wall)
- the goal is reached in about 1 episode in 70


**The task of Part B**

> **Raise the policy's expected return, using only sampled episodes like those.**

- the **expected return** of a policy is the average return over many episodes (made precise in the objective section below)
- the uniform policy's expected return is $-0.06$ (computed exactly, from the model)
- the best policy's expected return is $+0.35$: Part A found it by value iteration, **with the model**
- Part B has no model: it learns **from sampled episodes**
- the policy is the smallest network, the picture of Part A with a softmax output
    - input: the state as a one-hot vector of length $16$
    - one dense layer with $4$ outputs and a softmax: the probabilities of the 4 moves
    - this is multinomial logistic regression on the state
    - $\theta$ are its $16 \times 4 = 64$ parameters; all $0$ gives the uniform policy


## A second example: a language model is a policy

**Why this section?** FrozenLake is the example we compute with. The course ends at language models, and they fit the same loop.

> **A language model already is a policy: the prompt and the completion so far are the state, the next token is the action, and the whole completion is the episode.**

| | FrozenLake | Language model |
|:--|:--|:--|
| **state** $\stateseq_\tt$ | a cell | the prompt and the completion so far, $(\x, \y_{<\tt})$ |
| **action** $\actseq_\tt$ | a move | the next token $\y_\tt$, from the vocabulary $\Vocab$ |
| **policy** | 4 move probabilities | a probability per token |
| **episode** | moves until a hole or G | one whole completion |
| **reward** | $-0.01$ per move, $+1$ at G | usually one reward, at the end |

- a **token** is a word or part of a word from a fixed vocabulary
- the state is huge and the actions are the whole vocabulary, so the one-layer softmax gives way to a deep network
- the next state is the old state plus the chosen token: a completion's probability is the product of its token probabilities


## The objective: expected return

**Why this section?** Improving a policy means changing $\theta$. That needs a number to improve: the expected return.

> **$J(\theta)$ is the expected return: the average, over episodes generated by $\pi_\theta$, of the return from the initial state.**

$$
J(\theta) = \Exp{\traj \sim \pi_\theta} ( \rlret_{0, \traj} )
$$

- $\traj$ is one episode, and $\rlret_{0,\traj}$ is its return from step $0$ (the notation slide below explains the subscript)
- $\traj \sim \pi_\theta$ means episodes generated by following $\pi_\theta$
- this is the value of the initial state of *this* policy, $\statevalfun_{\pi_\theta}(\stateseq_0)$, from Part A
    - averaged over the distribution of starting states, if it is not fixed
- **for each fixed $\theta$, $\pi_\theta$ defines a distribution over episodes**
    - a different $\theta$ gives a different distribution, so $J$ is a function of $\theta$
    - $\theta$ is the argument of $J$ *and* sits inside the distribution we average over


**Gradient ascent on $J(\theta)$**

> **To improve the policy, move $\theta$ in the direction that raises $J$: add the gradient.**

$$
\theta_{k+1} = \theta_k + \alpha \, \nabla_\theta J(\theta_k)
$$

- $\nabla_\theta J(\theta)$ is the gradient of $J$ with respect to the parameters $\theta$
- $\alpha$ is the step size (learning rate)
- the update produces a sequence of parameters $\theta_0, \ldots, \theta_k, \ldots$ and so a sequence of policies $\pi_0, \ldots, \pi_k, \ldots$
- **ascent, not descent, because we maximize:** we add the gradient where a loss would subtract it
- source for the chapter: Sutton and Barto, *Reinforcement Learning*, Chapter 13


**Episodes and their probabilities**

> **Each episode counts in proportion to its probability under the current $\theta$: the policy $\pi_\theta$ decides which episodes are likely.**

$$
J(\theta) = \Exp{\traj \sim \pi_\theta} ( \rlret_{0, \traj} ) = \sum_\traj \prc{\traj}{\theta} \, \rlret_{0, \traj}
$$

- three probabilities appear in this session, and they are different things
    - $\transp$ (capital $P$): the environment's response, outside the agent's control
        - it can be random too: one observed $(\state', \rew)$ is a noisy sample, and a model-free method works from such samples
    - $\pi_\theta$: the policy, the agent's choice of action
    - $\prc{\traj}{\theta}$ (small $p$): the probability of a whole episode, which combines both
- change $\theta$ and the probabilities $\prc{\traj}{\theta}$ change, so $J$ changes
- from here each quantity names its episode $\traj$ as a subscript
    - the return: $\rlret_{\tt,\traj}$ (Part A: $\rlret_\tt$)
    - states, actions and rewards: $\stateseq_{\traj,\tt}$, $\actseq_{\traj,\tt}$, $\rewseq_{\traj,\tt+1}$
    - the return from step $\tt$: $\rlret_{\tt, \traj} = \rewseq_{\tt+1} + \disc \, \rlret_{\tt+1, \traj}$


## Why the gradient is not obvious

**Why this section?** $J$ is a sum over episodes, and each episode's probability involves the environment, which a model-free method does not know.

> **The question: how do we differentiate $J$, an average over episodes whose probabilities depend on $\theta$ and contain the unknown $\transp$?**

- the probability of an episode chains the policy and the environment at every step

$$
\prc{\traj}{\theta} = \pr{\stateseq_{\traj,0}} \prod_{\tt=0}^{|\traj|-1} \pi_\theta(\actseq_{\traj,\tt} \mid \stateseq_{\traj,\tt}) \; \transp(\stateseq_{\traj, \tt+1}, \rewseq_{\traj,\tt+1} \mid \stateseq_{\traj,\tt}, \actseq_{\traj,\tt})
$$

- two obstacles
    - $\theta$ sits inside the probabilities we average over, so we cannot just differentiate a sample average
    - the $\transp$ factors (and the start distribution) are **not available** in a model-free setting

<img src="images/b1_trajectory_chain.png" width=76%>


## Answering the question (1 of 2): the log-derivative trick

**Why this section?** We need $\nabla_\theta J$ as something we can estimate from samples. A sum of gradients is not an average, but a sum with each episode's probability in front of it is.

> **The log-derivative trick: for any probability $p$, $\ \nabla p = p \, \nabla \log p$.**
>
> The chain rule gives $\nabla \log p = \nabla p \,/\, p$. Multiply both sides by $p$.

- apply it to each episode's probability $\prc{\traj}{\theta}$

$$
\nabla_\theta J = \sum_\traj \nabla_\theta \prc{\traj}{\theta} \; \rlret_{0,\traj}
= \sum_\traj \underbrace{\prc{\traj}{\theta}}_{\text{probability}} \, \nabla_\theta \log \prc{\traj}{\theta} \; \rlret_{0,\traj}
= \Exp{\traj \sim \pi_\theta} \big[ \nabla_\theta \log \prc{\traj}{\theta} \; \rlret_{0,\traj} \big]
$$

- the first equality: the return of a fixed episode does not depend on $\theta$, so the gradient acts only on $\prc{\traj}{\theta}$
- the second: the trick, which puts the probability back in front
- the third: a sum weighted by probabilities is an expectation, which we can sample


**Answering the question (2 of 2): the transition probabilities drop out**

> **The log of a product is a sum, and every term without $\theta$ has gradient zero.**

$$
\log \prc{\traj}{\theta} = \log \pr{\stateseq_{\traj,0}} + \sum_{\tt=0}^{|\traj|-1} \Big[ \log \pi_\theta(\actseq_{\traj,\tt} \mid \stateseq_{\traj,\tt}) + \log \transp(\stateseq_{\traj,\tt+1}, \rewseq_{\traj,\tt+1} \mid \stateseq_{\traj,\tt}, \actseq_{\traj,\tt}) \Big]
$$

- the policy factors depend on $\theta$, and the start and $\transp$ factors do not
- the gradient of a sum is the sum of the gradients, so the start and $\transp$ terms drop out
- only $\sum_{\tt} \nabla_\theta \log \pi_\theta(\actseq_{\traj,\tt} \mid \stateseq_{\traj,\tt})$ remains
- each term is the gradient of the log-probability of an action taken: its **score** (next slide)


# The score of an action

**Why this section?** The derivation ended with the gradient of the log-probability of an action. It has a name, and the rest of Part B uses it.

> **The score function of an action** (the *score*, for short) **is the gradient of its log-probability with respect to the parameters: $\nabla_\theta \log \pi_\theta(\act \mid \state)$.**

- the score is a **vector with one entry per parameter**
    - entry $j$ says how much $\log \pi_\theta(\act \mid \state)$ rises if parameter $\theta_j$ rises
- so the score is the direction in parameter space that makes action $\act$ **more likely** in state $\state$
    - the log only makes the gradient convenient: it turned products into sums above
- the log-probability alone is not the score: the score is its *gradient*
- a **weight** in front of the score says how far to push
    - a positive weight makes the action more likely, a negative weight less likely


# The policy gradient theorem

**Why this section?** The derivation just gave the answer. State it as a theorem: the gradient can be written without the environment's probabilities.

> **Theorem: $\nabla_\theta J(\theta)$ is**
>
> - **an expectation** over episodes generated by $\pi_\theta$
> - **of** the **score** of each action taken, $\nabla_\theta \log \pi_\theta(\act \mid \state)$
> - **weighted by** the return of the episode

$$
\nabla_\theta J(\theta) = \underbrace{\Exp{\traj \sim \pi_\theta}}_{\text{expectation over episodes}} \left[ \sum_{\tt=0}^{|\traj|-1} \underbrace{\nabla_\theta \log \pi_\theta(\actseq_{\traj,\tt} \mid \stateseq_{\traj,\tt})}_{\text{score}} \; \underbrace{\rlret_{0,\traj}}_{\text{weight: the return}} \right]
$$

- **the transition probabilities $\transp$ do not appear**, so the gradient can be estimated from sampled episodes
- conditions: episodic tasks, $\pi_\theta$ differentiable in $\theta$, and the start distribution and $\transp$ independent of $\theta$
- every step's score is weighted by the whole episode's return
- other weights are the subject of the next sessions


**What one update does**

> **An action followed by a positive return becomes more likely. One followed by a negative return, less likely.**

- the setting: a policy over three actions, currently $\pi_\theta = (0.47,\ 0.32,\ 0.21)$
    - suppose the agent samples **action 2** and receives a return $\rlret$
- the update uses the score of the sampled action, weighted by the return
    - this is one term of the theorem's sum, for one sampled action

$$
\theta \leftarrow \theta + \alpha \; \rlret \; \nabla_\theta \log \pi_\theta(\act)
$$

- the score points toward parameters that make action 2 more likely
    - with $\rlret > 0$ we move that way: action 2 gains probability
    - with $\rlret < 0$ we move the opposite way: action 2 loses probability
    - the size of $\rlret$ sets how far

> **The sampled action's probability rises or falls; the probabilities of the other actions change the other way in total, since they sum to $1$.**

- caution: on FrozenLake most returns are near zero or negative, so this rule alone would push most actions down
    - next session fixes that with a reference value


**A fact we will use again: the zero-mean score lemma**

**What we are doing and why.** Next session subtracts a reference value from the return to reduce noise. This lemma says such a subtraction changes nothing on average.

> **Lemma. For every state $\state$, the score has mean zero over the action sampled from the policy:**
>
> $$\Exp{\act \sim \pi_\theta(\cdot \mid \state)} \big[ \nabla_\theta \log \pi_\theta(\act \mid \state) \big] = \sum_\act \pi_\theta(\act \mid \state) \, \nabla_\theta \log \pi_\theta(\act \mid \state) = 0$$

**Consequence.** Let $c$ be any quantity fixed before the action is chosen. Then

$$
\Exp{\act \sim \pi_\theta(\cdot \mid \state)} \big[ c \cdot \nabla_\theta \log \pi_\theta(\act \mid \state) \big] = c \cdot 0 = 0
$$

- in words: the probabilities must sum to $1$, so raising one action's probability lowers the others'
    - averaged over which action is sampled, these pushes cancel


**Why the lemma holds**

> **The mean of the score is the gradient of a constant: the probabilities always sum to $1$.**

$$
\begin{aligned}
\Exp{\act \sim \pi_\theta} \big[ \nabla_\theta \log \pi_\theta(\act \mid \state) \big]
&= \sum_\act \pi_\theta(\act \mid \state) \, \nabla_\theta \log \pi_\theta(\act \mid \state) && \text{definition of the expectation} \\
&= \sum_\act \nabla_\theta \pi_\theta(\act \mid \state) && \text{since } p \, \nabla \log p = \nabla p \\
&= \nabla_\theta \sum_\act \pi_\theta(\act \mid \state) && \text{the gradient of a sum is the sum of the gradients} \\
&= \nabla_\theta 1 = 0 && \text{the probabilities sum to } 1
\end{aligned}
$$

- so a quantity $c$ fixed before the action, times the score, has mean zero
    - subtracting such a $c$ leaves the average gradient unchanged
    - it changes only the **variance**: how much the estimate fluctuates between samples


## From the theorem to an algorithm: sample episodes

**Why this section?** The theorem is an expectation over episodes, and we cannot sum over all episodes. Part A met the same problem and used the same answer: sample.

> **The theorem averages over *all* possible episodes, which we cannot enumerate. Replace that expectation by an average over $N$ *sampled* episodes: the result is the estimator of **REINFORCE (over-simplified)**, the simplest policy-gradient method.**

- the weight is the return of the whole episode, with no baseline
- B2 builds up to Sutton and Barto's REINFORCE (return from each step) and REINFORCE with baseline (sections 13.3 and 13.4)

- the same move as in Part A, where Monte Carlo averaged sampled returns instead of summing over all trajectories
- here we run the current policy $\pi_\theta$ to generate $N$ episodes, and average

$$
\widehat{\nabla_\theta J}(\theta) = \frac{1}{N} \sum_{i=1}^{N} \sum_{\tt=0}^{|\traj_i|-1} \nabla_\theta \log \pi_\theta(\actseq_{i,\tt} \mid \stateseq_{i,\tt}) \, \rlret_{0,\traj_i}
$$

- then take a step $\theta \leftarrow \theta + \alpha \, \widehat{\nabla_\theta J}(\theta)$, where the hat marks the sampled *estimate*
- the environment's effect enters only through the sampled rewards and next states: the model $\transp$ is never needed


**What one sampled estimate looks like on FrozenLake**

> **The estimate is unbiased, but one episode's estimate is very noisy: for the entry shown, its spread is about 30 times the exact value.**

<img src="images/fl_gradient_noise.png" width=82%>

- the exact gradient (green) is computed from the model
- each sampled estimate uses one episode
- their mean is the exact value: the estimate is **unbiased**
- but a single estimate can be far off, even in the wrong direction
- B2 asks what can shrink the spread without moving the mean


**This estimator is on-policy**

> **The episodes must come from the policy being improved: the estimator is on-policy.**

- after each step the policy changes, so the old episodes are used up
- deep Q-learning reused old transitions because Q-learning is off-policy
- PPO's probability ratio (B4) will let one batch of episodes serve several updates


## Preview: one pattern, three slots

**Why this section?** Part B presents several methods: REINFORCE, actor-critic and PPO (GRPO in Part C).

They share one pattern, so each is a substitution in the estimator we just built.

> **Every method in Part B is the estimator we just built, with a substitution in up to three places (slots). Each later session fills one of them.**

<img src="images/b1_three_slots.png" width=62%>

- **slot 1, the weight:** the number that multiplies each score (here, the raw return)
- **slot 2, how far:** the step size of the update (here, a fixed $\alpha$)
- **slot 3, the baseline:** a reference value subtracted from the return (here, none)
- slot 3 is part of slot 1: the weight is the return minus the baseline
- later sessions only say which slot they fill: B2 and B3 fill slots 1 and 3, B4 fills slot 2


## The appendix

**Why this section?** The main line proved the theorem. The appendix proves two further results, which the next session uses.

> **We recommend reading the appendix** (`RL_B1_Appendix_PG_proofs.ipynb`) **before the next session.**

- the appendix holds
    - the zero-mean score lemma, and how it applies inside an episode: condition on the history, then average over the action
    - why weighting each step by the return from that step on, instead of the whole episode's return, gives the same expected gradient
    - an optional second route to that result, from per-step rewards
- the lemma's statement is all the next session needs
- the appendix shows how it applies inside an episode


# What we established, and what it sets up

- **Established**
    - a policy network outputs a distribution (softmax for discrete actions, mean and spread for continuous)
    - the agent samples from it, so exploration is built in
    - the **score** of an action is the gradient of its log-probability: a vector with one entry per parameter
    - $J(\theta)$ is the expected return, and we climb it by gradient ascent
    - **the gradient is an expectation over episodes of the score times a return**
        - the transition probabilities $\transp$ do not appear, so it can be estimated from sampled episodes
    - **the score has mean zero**, so a term fixed before the action adds no bias
    - the estimator is **on-policy**, and the raw return makes it noisy
    - the proofs are in the appendix (recommended)


**What it sets up**

- next session: the raw return is a noisy weight (high variance)
- baselines and advantages reduce it, using the lemma
- a learned value can supply the baseline (actor-critic)
- then how far one update may move (PPO), then in Part C a baseline without a critic (GRPO)
